# 🚨 The Masterclass: Anomaly Detection & Association Rules

We are moving into the final frontiers of Unsupervised Learning.

### 1. Anomaly Detection (Outliers)
Instead of clustering things into groups, what if we want to find the **freaks of nature**? 
In biology, you might have 5,000 normal cells and exactly 3 cells that have undergone a catastrophic mutation. K-Means will fail here. We need specialized algorithms like **Isolation Forest** to hunt down the 3 mutant cells.

### 2. Association Rules (Gene Linkage)
Famously known as "Market Basket Analysis" (If you buy bread, do you buy butter?).
In bioinformatics, this translates to: *"If a patient has mutation A and mutation B, what is the mathematical probability they will also develop mutation C?"*

Let's dive into the mathematics of both.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

sns.set_theme(style="darkgrid")
print("Libraries loaded. Ready for Outliers and Associations!")

---
## 🌲 Part 1: Anomaly Detection (The Isolation Forest)

**The Intuition:**
Normal data points are packed tightly together in the center. Anomaly data points are far out in the middle of nowhere.

**How Isolation Forest works:**
1. The AI randomly drops a straight line (a boundary) across the graph.
2. It drops another random line. And another.
3. To trap a *normal* cell inside its own box, the AI has to drop hundreds of lines because the cells are so crowded.
4. To trap an *anomalous mutated* cell, it only takes 1 or 2 lines because the cell is completely alone!

**The Math Rule:** Short path length (easy to isolate) = Anomaly. Long path length (hard to isolate) = Normal.

In [ ]:
# Let's synthesize a giant cluster of Healthy Cells
np.random.seed(42)
healthy_cells = np.random.normal(loc=0, scale=1.0, size=(500, 2))

# Let's inject 15 dangerously mutated cells (their gene expressions are extreme)
mutated_cells = np.random.uniform(low=-5, high=5, size=(15, 2))

# Combine them all into one raw biological dataset
X_biology = np.vstack([healthy_cells, mutated_cells])

plt.figure(figsize=(9, 6))
plt.scatter(X_biology[:,0], X_biology[:,1], alpha=0.5, color="gray")
plt.title("The Raw Data: Can you spot the outliers?")
plt.xlim(-6, 6)
plt.ylim(-6, 6)
plt.show()

### 🎯 Deploying the Isolation Forest
We use `sklearn.ensemble.IsolationForest`.
The crucial hyperparameter is `contamination`. If we suspect 3% of our blood sample is cancerous, we set `contamination=0.03`.

In [ ]:
# 1. Initialize the Forest (expecting around 3% anomalies)
iso_forest = IsolationForest(contamination=0.03, random_state=42)

# 2. Predict! 
# IMPORTANT: IsolationForest returns 1 for NORMAL, and -1 for ANOMALY.
predictions = iso_forest.fit_predict(X_biology)

# Visualize the Hunt
plt.figure(figsize=(10, 7))

# We use a dictionary to strictly map the colors
sns.scatterplot(
    x=X_biology[:,0], 
    y=X_biology[:,1], 
    hue=predictions, 
    palette={1: "#3498db", -1: "#e74c3c"}, 
    alpha=0.8, 
    s=60
)

plt.title("Isolation Forest Results: Red (-1) = Anomalies, Blue (1) = Healthy")
plt.xlim(-6, 6)
plt.ylim(-6, 6)
plt.legend(title="Cell Status")
plt.show()

# It perfectly identified the rogue extreme cells roaming around the edges!

---
## 🔗 Part 2: Association Rules (Apriori & FP-Growth)

Association rules don't care about numbers or distances. They only care about **Co-occurrence** (Things happening together).

### The 3 Master Math Equations:
Imagine we are checking if Gene Mutation A (`BRCA1`) is linked to Gene Mutation B (`TP53`).

1. **Support:** How often do A and B happen together in the whole hospital? 
   $$ \text{Support(A & B)} = \frac{\text{Patients with A and B}}{\text{Total Patients}} $$
2. **Confidence:** If a patient has A, what are the odds they also have B?
   $$ \text{Confidence(A} \rightarrow \text{B)} = \frac{\text{Support(A & B)}}{\text{Support(A)}} $$
3. **Lift:** Is this a literal biological link, or just random coincidence? 
   * Lift = 1: Total random coincidence.
   * Lift > 1: Very strong actively linked mutation!
   * Lift < 1: They actually repel each other (One gene suppresses the other).
   $$ \text{Lift(A} \rightarrow \text{B)} = \frac{\text{Confidence(A} \rightarrow \text{B)}}{\text{Support(B)}} $$

Let's write a pure Python implementation to see the math in action on a small Genomic Matrix.

In [ ]:
# A mock dataset of 5 Patients. 1 means they have the mutation, 0 means they don't.
patient_data = {
    'Patient': ['Patient_1', 'Patient_2', 'Patient_3', 'Patient_4', 'Patient_5'],
    'BRCA1': [1, 1, 0, 1, 0],
    'TP53':  [1, 1, 0, 0, 1],
    'EGFR':  [0, 0, 1, 1, 1]
}
df = pd.DataFrame(patient_data).set_index('Patient')

print("--- Genomic Mutation Matrix ---")
print(df)

Let's calculate the exact Association Rule mapping: **Does having a `BRCA1` mutation lead to a `TP53` mutation?**

In [ ]:
N = len(df) # Total patients = 5

# 1. Calculate SUPPORT
both_happen = len(df[(df['BRCA1'] == 1) & (df['TP53'] == 1)])
support = both_happen / N
print(f"SUPPORT (BRCA1 & TP53): {both_happen} out of {N} patients = {support*100}% frequency.")

# 2. Calculate CONFIDENCE
brca1_total = len(df[df['BRCA1'] == 1])
confidence = both_happen / brca1_total
print(f"CONFIDENCE (BRCA1 -> TP53): Out of {brca1_total} BRCA1 patients, {confidence*100:.1f}% also have TP53.")

# 3. Calculate LIFT
tp53_total = len(df[df['TP53'] == 1])
support_brca1 = brca1_total / N
support_tp53 = tp53_total / N

lift = support / (support_brca1 * support_tp53)
print(f"\nLIFT MULTIPLIER: {lift:.2f}")

if lift > 1:
    print("✅ LIFT > 1 ! These two mutations are actively linked and trigger each other!")
elif lift < 1:
    print("❌ LIFT < 1 ! These two mutations repel each other (One protects against the other).")
else:
    print("〰️ LIFT = 1 ! They have zero effect on each other. Pure random chance.")

### 🎉 Conclusion

**Anomaly Detection:** Is incredible for finding the 0.01% of data that doesn't belong (Fraud detection, catastrophic cancer cells, weird microscope artifacts). 
* `Isolation Forest` creates random boundaries.
* `One-Class SVM` tries to draw a tight mathematical circle around your normal data and flags anything that steps outside the boundary.

**Association Rules:** Used anytime you have pure "Basket" data. It's not about numbers or sizes, it's strictly about *"If A happens, will B happen?"* Algorithms like `Apriori` simply run those exact 3 Support/Confidence/Lift formulas across millions of rows at lightning speed to find the hidden links.